In [12]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import label_ranking_average_precision_score
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier

# 1. Load Data
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

# 2. Encode Categorical Features
cat_features = ['Soil Type', 'Crop Type']
for col in cat_features:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
    test[col] = le.transform(test[col])

# 3. Encode Target
target_le = LabelEncoder()
train['Fertilizer Name'] = target_le.fit_transform(train['Fertilizer Name'])

# 4. Features/Target
features = [col for col in train.columns if col not in ['id', 'Fertilizer Name']]
X = train[features]
y = train['Fertilizer Name']
X_test = test[features]

# 5. Validation MAP@3 (optional)
params = {
    'objective': 'multiclass',
    'num_class': len(target_le.classes_),
    'metric': 'None',
    'learning_rate': 0.05,
    'verbosity': -1,
    'seed': 42,
}
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
lgb_train = lgb.Dataset(X_tr, y_tr)
lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)
model = lgb.train(params, lgb_train, num_boost_round=200)
val_probs = model.predict(X_val)
map3 = label_ranking_average_precision_score(
    np.eye(val_probs.shape[1])[y_val], val_probs
)
print(f"Validation MAP: {map3:.4f}")

Validation MAP: 0.4177


In [13]:
# --- Gradient Boosting ---
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.05,
    random_state=42
)
gb_model.fit(X, y)
gb_probs = gb_model.predict_proba(X_test)

In [14]:

# --- Get Top 3 Predictions ---
top3 = np.argsort(gb_probs, axis=1)[:, -3:][:, ::-1]
top3_flat = top3.flatten()
top3_names_flat = target_le.inverse_transform(top3_flat)
top3_names = top3_names_flat.reshape(top3.shape)
top3_str = [' '.join(row) for row in top3_names]
top3_str

['28-28 DAP 20-20',
 '17-17-17 20-20 14-35-14',
 '10-26-26 14-35-14 20-20',
 '14-35-14 10-26-26 17-17-17',
 '20-20 10-26-26 17-17-17',
 '28-28 17-17-17 10-26-26',
 '10-26-26 14-35-14 17-17-17',
 '17-17-17 14-35-14 10-26-26',
 '20-20 14-35-14 17-17-17',
 '10-26-26 14-35-14 17-17-17',
 '17-17-17 14-35-14 28-28',
 '28-28 14-35-14 20-20',
 '14-35-14 17-17-17 20-20',
 '17-17-17 10-26-26 14-35-14',
 '17-17-17 10-26-26 28-28',
 '17-17-17 14-35-14 20-20',
 '10-26-26 28-28 14-35-14',
 '20-20 14-35-14 17-17-17',
 '14-35-14 20-20 28-28',
 '14-35-14 10-26-26 17-17-17',
 '17-17-17 14-35-14 10-26-26',
 '28-28 20-20 14-35-14',
 '10-26-26 14-35-14 20-20',
 '20-20 10-26-26 14-35-14',
 '28-28 17-17-17 14-35-14',
 '10-26-26 17-17-17 20-20',
 '28-28 20-20 10-26-26',
 'DAP 10-26-26 28-28',
 '17-17-17 10-26-26 14-35-14',
 '17-17-17 28-28 14-35-14',
 '10-26-26 20-20 14-35-14',
 '14-35-14 28-28 10-26-26',
 '20-20 10-26-26 28-28',
 '17-17-17 10-26-26 14-35-14',
 '28-28 14-35-14 10-26-26',
 '28-28 17-17-17 14-3

In [ ]:

# --- Prepare Submission ---
submission = pd.DataFrame({
    'id': test['id'],
    'Fertilizer Name': top3_str
})
submission.to_csv('submission.csv', index=False)
print("Submission file created: submission.csv")